# Initialization

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

from skopt import gp_minimize
from skopt.space import Real, Categorical
from skopt.learning import GaussianProcessRegressor
from skopt.learning.gaussian_process.kernels import Matern, RBF

import numpy as np
import pandas as pd
from pathlib import Path

## Directories

In [2]:
data_dir = Path("data")
Path.mkdir(data_dir, exist_ok=True)

plot_dir = Path("plots")
Path.mkdir(plot_dir, exist_ok=True)

log_dir = Path("logs")
Path.mkdir(log_dir, exist_ok=True)

## Data reading

In [3]:
data_file = "Dataset_SL.xlsx"

### Define experimental data

In [4]:
# 1. Define the experimental data
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")

In [5]:
# make the other columns as floats
experiment_data = experiment_data.astype(
    {
        "salt_concentration": float,
        "water_to_cement_ratio": float,
        "antisettling_concentration": float,
        "E_d": float,
        "KPI": float,
    }
)

# get optimization round for later use in saving
opt_round = experiment_data["opt_round"].iloc[-1]

In [ ]:
experiment_data

# Search space

### Category encoding

In [ ]:
category_mapping = {
    "Al2(SO4)3": 0,
    "CaCl2": 1,
    "CuSO4": 2,
    "K2CO3": 3,
    "KAl(SO4)2": 4,
    "LiCl": 5,
    "Mg(NO3)2": 6,
    "MgCl2": 7,
    "MgSO4": 8,
    "SrBr2": 9,
    "Zn(NO3)2": 10,
}

df_enc = experiment_data.copy()
df_enc["category_encoded"] = experiment_data["salt"].map(category_mapping)
df_enc

In [19]:
search_space = [
    Categorical(list(category_mapping.keys()), name="salt"),  # salt names
    Real(0.1, 0.9, name="salt_concentration"),
    Real(0.7, 1.5, name="water_to_cement_ratio"),
    Real(0.0, 3.0, name="antisettling_concentration"),
]

# Optimization

In [9]:
salt_dummies = pd.get_dummies(df_enc["salt"], prefix="salt")

X = pd.concat(
    [
        salt_dummies,
        df_enc[
            [
                "salt_concentration",
                "water_to_cement_ratio",
                "antisettling_concentration",
            ]
        ],
    ],
    axis=1,
).to_numpy()
y_energy = df_enc["E_d"].values
y_kpi = df_enc["KPI"].values

In [ ]:
salt_dummies

## Gaussian Process

In [11]:
def fit_gp_models(X, y, kernel):
    """Fits Gaussian Process models to the given experimental data."""  # noqa

    gp = GaussianProcessRegressor(
        kernel=kernel, normalize_y=True, n_restarts_optimizer=10
    )

    gp.fit(X, y)

    return gp

### Matern kernel

In [12]:
gp_energy_matern = fit_gp_models(
    X,
    -y_energy,
    Matern(length_scale=5e-3, length_scale_bounds=(1e-8, 0.5), nu=2.5),  # noqa
)

In [ ]:
gp_kpi_matern = fit_gp_models(
    X,
    y_kpi,
    Matern(length_scale=5e-3, length_scale_bounds=(1e-8, 0.5), nu=2.5),  # noqa
)

### RBF kernel

In [ ]:
gp_energy_rbf = fit_gp_models(
    X,
    -y_energy,
    RBF(
        length_scale=5e-3,
        length_scale_bounds=(1e-8, 0.5),
    ),
)

In [ ]:
gp_kpi_rbf = fit_gp_models(
    X,
    y_kpi,
    RBF(
        length_scale=5e-3,
        length_scale_bounds=(1e-8, 0.5),
    ),
)

# Bayesian Optimization

In [16]:
def encode_input(x):
    # x[0] is salt name (e.g., "MgSO4")
    salt_vector = np.zeros(len(category_mapping))
    salt_index = category_mapping[x[0]]
    salt_vector[salt_index] = 1

    # concatenate with continuous features
    return np.concatenate([salt_vector, np.array(x[1:])])

In [17]:
def suggest_new_samples(
    gp_model: GaussianProcessRegressor,
    obj_func: str,
    verbose: bool = False,
):
    """Suggests new samples based on the trained GP model using different acquisition functions."""  # noqa

    acq_functions = ["EI", "PI", "LCB_low", "LCB_med", "LCB_high"]
    kappa_values = {
        "LCB_low": 1.0,
        "LCB_med": 2.5,
        "LCB_high": 5.0,
    }
    new_samples = []

    for i, acquisition in enumerate(acq_functions):
        res = gp_minimize(
            lambda x: gp_model.predict([encode_input(x)])[
                0
            ],  # Optimize our surrogate model # noqa
            dimensions=search_space,  # pass our search space
            base_estimator=gp_model,  # ask for our custom estimators
            acq_func=(
                "LCB"
                if acquisition in ["LCB_low", "LCB_med", "LCB_high"]
                else acquisition
            ),  # define the acquisition function
            kappa=kappa_values.get(
                acquisition, 1.96
            ),  # define the custom k value if acquisition if "LCB" # noqa
            xi=0.01,
            n_calls=20,
            n_initial_points=10,
            initial_point_generator="random",
        )

        # Convert numerical salt encoding back to categorical
        suggested = res.x
        if verbose:
            print(suggested)

        suggested.append(str(gp_model.kernel).split("(")[0])
        suggested.append(str(acquisition))
        suggested.append(str(obj_func))

        # save suggested samples in a list
        new_samples.append(suggested)

    # create a new dataframe with suggested samples
    columns = [
        "salt",
        "salt_concentration",
        "water_to_cement_ratio",
        "antisettling_concentration",
        "kernel",
        "acquisition",
        "obj_func",
    ]

    df = pd.DataFrame(new_samples, columns=columns)

    # round float values to 3rd decimal place
    df[df.select_dtypes(include="float").columns] = df.select_dtypes(
        include="float"
    ).round(3)

    return df

### Generate 10 new samples for Energy Density optimization

In [ ]:
# Matérn kernel
new_samples_energy_matern = suggest_new_samples(
    gp_energy_matern, obj_func="E_d", verbose=True
)

In [ ]:
new_samples_energy_matern

In [ ]:
# RBF kernel
new_samples_energy_rbf = suggest_new_samples(
    gp_energy_rbf, obj_func="E_d", verbose=True
)

In [ ]:
new_samples_energy_rbf

### Generate 10 new samples for economic KPI optimization

In [ ]:
# Matérn kernel
new_samples_kpi_matern = suggest_new_samples(
    gp_kpi_matern, obj_func="KPI", verbose=True
)

In [ ]:
new_samples_kpi_matern

In [ ]:
# RBF kernel
new_samples_kpi_rbf = suggest_new_samples(
    gp_kpi_rbf, obj_func="KPI", verbose=True
)  # noqa

In [ ]:
new_samples_kpi_rbf

## Saving the new suggestions in the original excel sheet

In [ ]:
# Combine sets of new samples
new_samples = pd.concat(
    [
        new_samples_energy_matern,
        new_samples_energy_rbf,
        new_samples_kpi_matern,
        new_samples_kpi_rbf,
    ],
    ignore_index=True,
)

new_samples.insert(0, "opt_round", opt_round + 1)  # add optimization round

# Read the existing Excel sheet into a DataFrame
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")
# Append the new data to the existing DataFrame
combined_data = pd.concat(
    [experiment_data, new_samples], ignore_index=True, axis=0
)  # noqa

# Write the updated DataFrame back to the same Excel sheet
with pd.ExcelWriter(
    data_dir / data_file, engine="openpyxl", mode="a", if_sheet_exists="replace"  # noqa
) as writer:
    combined_data.to_excel(writer, sheet_name="Datasheet", index=False)

print(
    "New batch of 20 samples suggested. Please conduct experiments and update the dataset."  # noqa
)